# Amparo -- Baseline y fine-tuning LoRA del agente legal

Este notebook:
1. Carga `dataset_m1_v2.jsonl` (**2291 ejemplos**: 1536 de v1 sin contexto y 755 de v2 con contexto recuperado) y usa el split que ya viene resuelto en el archivo. La validacion de v1 es la misma de siempre, asi que las cifras se pueden comparar con la corrida de M2 del 2026-10-06.
2. Corre el modelo base sin modificar sobre esa validacion (**baseline**).
3. Entrena un adaptador **LoRA** sobre el resto del dataset.
4. Vuelve a correr el modelo ya afinado sobre la misma validacion.
5. Compara baseline vs. afinado -- esa comparacion es la evidencia real de que el fine-tuning mejoro algo, no solo una suposicion.

**Sobre esta version del dataset.** La version anterior (1320 ejemplos) daba consejo generico sin decirle a la persona a donde acudir: solo el 14% de las respuestas nombraba un mecanismo legal concreto, lo que contradecia el principio 3 de `PRODUCT.md`. El dataset se reescribio para que cada respuesta señale la ruta real (tutela, Inspeccion del Trabajo, Comisaria de Familia, centro de conciliacion, etc.) y la evidencia a reunir; hoy lo cumple el **99.7%**. Tambien se agregaron tres categorias que faltaban y concentraban los casos de urgencia: servicios publicos domiciliarios, violencia intrafamiliar y penal basico/victimas.

**Que cambio con v2.** El dataset original no ensenaba numeros de articulo a proposito: ese conocimiento lo aporta el RAG de M3. Pero en la corrida del 2026-10-07 el modelo cito en **0 de 9** casos aun teniendo el articulo correcto delante (`results/m3_s08_2026-10-07`), porque nunca vio un ejemplo con contexto: 0 de 1536. Los 755 ejemplos de v2 ensenan la conducta que faltaba -- usar el CONTEXTO y decir de que norma sale -- sin tocar v1, que sigue ensenando a responder sin inventar cuando no hay contexto. Los dos modos traen system prompts distintos, asi que el modelo aprende la conducta condicional.

Antes de correr: `Entorno de ejecucion > Cambiar tipo de entorno de ejecucion > GPU` (una T4 gratis alcanza usando 4-bit/QLoRA).

**Nota sobre las librerias:** `transformers`, `peft` y `trl` cambian su API con cierta frecuencia. Este notebook esta escrito contra las versiones estables mas recientes conocidas al momento de escribirlo; si algun cambio de firma da error, revisa el changelog de la libreria correspondiente -- la logica de cada celda (que hace y por que) sigue siendo valida aunque cambie algun nombre de parametro.

In [1]:
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets wandb
!pip install -U "bitsandbytes>=0.46.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.5/925.5 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.1/26.1 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 31.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires py

## Codigo y dataset desde el repo

In [2]:
# El codigo y el dataset se toman del repo, no de Drive.
#
# Antes el dataset se leia de Drive y las metricas vivian copiadas dentro del
# notebook. Las dos cosas se desincronizaron con consecuencias reales: una
# version recortada del catalogo de rutas marcaba como "no nombra ruta"
# respuestas que si la nombraban, y nada fallaba a la vista. Clonando el repo
# hay una sola fuente de verdad para el dataset y para cada metrica.
RAMA = "m3.5"

import os

if not os.path.isdir("/content/Amparo"):
    !git clone -q -b {RAMA} https://github.com/TomasPosada0626/Amparo.git
else:
    !cd /content/Amparo && git fetch -q origin && git checkout -q {RAMA} && git pull -q
%cd /content/Amparo

!git log -1 --format="Codigo del commit %h (%ad)" --date=short


/content/Amparo
Codigo del commit ba91819 (2026-10-05)


## Configuracion

Cambia `MODEL_ID` para comparar Qwen2.5 7B vs. Llama 3.1 8B (o el que haya ganado en el comparador local `tools/model_comparator`).

In [3]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"  # alternativa: "meta-llama/Llama-3.1-8B-Instruct"

VAL_FRACTION = 0.15   # % por categoria reservado para validacion (nunca se entrena con esto)
RANDOM_SEED = 42

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
PER_DEVICE_BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 8   # batch efectivo = PER_DEVICE_BATCH_SIZE * GRAD_ACCUM_STEPS
# 1024 alcanzaba cuando el prompt era system + pregunta. Los ejemplos de v2
# traen ademas el CONTEXTO (hasta 5 fragmentos de norma), asi que a 1024 se
# truncarian justo por donde esta lo que deben ensenar a citar.
MAX_SEQ_LENGTH = 3072

# Era 300: a ese techo el baseline (que responde largo, con listas) quedaba
# cortado a mitad de frase en la mayoria de los casos, y la evaluacion lo
# castigaba por incompleto en vez de por incorrecto. 900 es el mismo valor
# que MAX_NEW_TOKENS_GENERATION de tools/evaluation/config.py, para que la
# cifra de M1 y la de M2 se midan bajo las mismas condiciones.
MAX_NEW_TOKENS_EVAL = 900
OUTPUT_DIR = "/content/amparo-lora"

## Monta tu Google Drive (solo para las salidas)

El dataset ya NO se sube a Drive: viene del repo clonado arriba, que es la
fuente de verdad. Antes se leia de Drive y eso permitia entrenar con una copia
vieja sin ningun error visible.

En Drive quedan solo las cosas que deben sobrevivir al reinicio de la maquina
de Colab: el adaptador entrenado y los resultados de la evaluacion.

El dataset no se edita a mano: se construye desde las fuentes versionadas en
`data/dataset_src/` con `python -m tools.dataset_build`, que solo lo escribe si
pasa las ocho puertas de calidad de `tools/dataset_quality.py`. Asi queda
trazable con que datos se entreno y cualquier cambio es revisable ejemplo por
ejemplo en un diff.


In [4]:
from google.colab import drive

drive.mount('/content/drive')

# El dataset sale del repo clonado (fuente de verdad). En Drive quedan solo las
# SALIDAS, que si deben sobrevivir al reinicio de la maquina de Colab.
# El combinado v1+v2 (tools/dataset_m1_v2.py): 1536 ejemplos sin contexto mas
# 755 con contexto, cada uno con su origen y su split ya resueltos.
DATASET_PATH = '/content/Amparo/data/dataset_m1_v2.jsonl'

DRIVE_DIR = '/content/drive/MyDrive/Colab Notebooks/Amparo'
SALIDA_DIR = f'{DRIVE_DIR}/evaluacion'   # resultados de la corrida, no en /content
# Con fecha: el adaptador sin version se sobrescribio el 6 de octubre y las
# corridas del 27 de septiembre quedaron sin forma de reproducirse. La carpeta
# sin fecha se sigue actualizando porque tools/rag/config.py apunta ahi, pero
# la fechada queda como respaldo.
from datetime import date
VERSION_ADAPTADOR = date.today().isoformat()
ADAPTER_DIR = f'{DRIVE_DIR}/amparo-lora-adapter'
ADAPTER_DIR_VERSIONADO = f'{DRIVE_DIR}/amparo-lora-adapter-{VERSION_ADAPTADOR}'

import os
os.makedirs(SALIDA_DIR, exist_ok=True)
print(f'Dataset (del repo): {DATASET_PATH}')
print(f'Resultados -> {SALIDA_DIR}')


Mounted at /content/drive
Dataset (del repo): /content/Amparo/data/dataset_legal.jsonl
Resultados -> /content/drive/MyDrive/Colab Notebooks/Amparo/evaluacion


## Cargar y separar train / validacion (estratificado por categoria)

In [5]:
import json
import random
from collections import Counter, defaultdict

random.seed(RANDOM_SEED)

records = [json.loads(line) for line in open(DATASET_PATH, encoding='utf-8')]

by_category = defaultdict(list)
for r in records:
    by_category[r['category']].append(r)

# El split viene resuelto en el archivo (tools/dataset_m1_v2.py) y NO se
# recalcula aca. v1 se partio con stratified_split -- verificado que da la misma
# validacion de 231 que hacia este notebook, asi que las cifras se siguen
# pudiendo comparar con M2 del 2026-10-06 -- y v2 con split_v2, que manda cada
# ejemplo al lado de su pregunta base: la misma pregunta sin contexto en train y
# con contexto en val haria que val midiera memoria.
train_records = [r for r in records if r['split'] == 'train']
val_records = [r for r in records if r['split'] == 'val']
random.shuffle(train_records)
random.shuffle(val_records)

print(f'Total: {len(records)} | Train: {len(train_records)} | Validacion: {len(val_records)}')
# Guarda contra el fallo silencioso de entrenar con una copia vieja del dataset
# que haya quedado en Drive: sin esto, el notebook corre igual y nadie se entera.
assert len(records) == 2291, (
    f'El dataset tiene {len(records)} ejemplos y se esperaban 2291 (1536 de v1 '
    f'y 755 de v2). Regeneralo con: python -m tools.dataset_m1_v2'
)
_origenes = Counter(r['origen'] for r in records)
assert _origenes == {'v1': 1536, 'v2': 755}, (
    f'Mezcla inesperada de origenes: {dict(_origenes)}. Sin los ejemplos de v2 '
    f'el modelo no aprende a usar el contexto, que es el punto de esta corrida.'
)
_modos = Counter(r.get('modo') for r in records if r['origen'] == 'v2')
print(f'  v1 {_origenes["v1"]} sin contexto | v2 {_origenes["v2"]} con contexto {dict(_modos)}')

for category in sorted(by_category):
    n_val = sum(1 for r in val_records if r['category'] == category)
    n_train = sum(1 for r in train_records if r['category'] == category)
    print(f'  {category:45s} train={n_train:3d}  val={n_val:3d}')

Total: 1536 | Train: 1305 | Validacion: 231
  Acceso a informacion publica                  train= 43  val=  8
  Accidentes de transito                        train= 52  val=  9
  Arriendo                                      train= 52  val=  9
  Comparendos de transito                       train= 52  val=  9
  Conciliacion prejudicial                      train= 43  val=  8
  Contratacion estatal y facturacion            train= 43  val=  8
  Contratos empresariales (B2B)                 train= 43  val=  8
  Derecho administrativo general                train= 43  val=  8
  Derecho ambiental sancionatorio               train= 43  val=  8
  Derecho contractual general                   train= 43  val=  8
  Derecho de familia - alimentos                train= 43  val=  8
  Despido                                       train= 53  val=  9
  Educacion / debido proceso disciplinario      train= 43  val=  8
  Embargos                                      train= 52  val=  9
  Fuera del derech

## Cargar el modelo base en 4-bit (QLoRA)

Se carga cuantizado en 4-bit para que quepa comodamente en una GPU gratuita de Colab (T4, ~16GB).

In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

## Metricas de dominio

Se importan del repo (`tools/`), no van copiadas aqui: una sola definicion por
propiedad, la misma que usan las puertas del dataset y el harness de M2.

- **Nombra ruta legal** (`tools/dataset_quality.py`): principio 3 de PRODUCT.md.
- **Cita normas inventadas** (`tools/evaluation/domain_metric.py`): la misma
  definicion que M2, para que las cifras sean comparables.
- **Inventa entidades** (`tools/evaluation/entity_metric.py`): el segundo modo
  de fabricacion, al que el regex de citas es ciego porque no lleva numero.

**Por que esta cifra puede no coincidir con la de M2.** Aqui se mide el modelo
que quedo en memoria despues de entrenar; M2 recarga el adaptador desde Drive
sobre el modelo base, normalmente en otra GPU. Con decodificacion greedy
cualquier diferencia numerica hace que el texto diverja desde el primer token
ambiguo. Las dos mediciones son legitimas pero no identicas, y la de referencia
es la de M2, por estar respaldada por un artefacto comiteado con su manifiesto.


In [7]:
import json
import re

# Las metricas se IMPORTAN del repo: una sola definicion por propiedad.
#   - nombra_mecanismo: el mismo catalogo con el que se validaron las puertas
#     del dataset (tools/dataset_quality.py).
#   - cita_no_verificable: la misma definicion que usa el harness de M2
#     (tools/evaluation/domain_metric.py), para que las cifras sean comparables.
#   - entidades_inventadas: el segundo modo de fabricacion
#     (tools/evaluation/entity_metric.py).
#
# Antes estas definiciones iban copiadas dentro del notebook y la copia se
# desincronizo del repo, subestimando al modelo sin que nada fallara.
from tools.dataset_quality import menciona_mecanismo as nombra_mecanismo
from tools.evaluation.domain_metric import has_invented_citation as cita_no_verificable
from tools.evaluation.entity_metric import find_fabricated_entities

SYSTEM_PROMPT = records[0]['messages'][0]['content']

# --- Metricas de dominio ---------------------------------------------------
# Miden lo que el producto promete y son verificables sin juez LLM.
#
# 1) Nombrar la ruta legal concreta (principio 3 de PRODUCT.md). OJO al leerla:
#    esta saturada (el modelo base ya venia en 96%) y esta confundida con la
#    longitud, porque es coincidencia de palabras clave y el baseline escribe
#    ~5x mas texto, o sea ~5x mas superficie para acertar por azar. No sirve
#    para demostrar mejora en ninguna direccion.
# 2) Citar normas por numero. El dataset no las ensena a proposito -- ese
#    conocimiento lo aporta el RAG de M3 contra el corpus real -- asi que toda
#    cita que aparezca sale de la memoria del modelo.
# 3) Inventar entidades. Aparecio midiendo la corrida del 2026-10-05: el
#    adaptador dejo de inventar normas y seguia mandando a ventanillas que no
#    existen ("Superintendencia de Pensiones", "querella de omision"). Es el
#    mismo pecado y el regex de citas es ciego a el porque no llevan numero.
#
# Ya NO se calcula similitud lexica con difflib: daba 3.2 contra 6.0 sobre 100,
# no distingue una respuesta correcta de una incorrecta (solo fraseo) y la
# comparacion contra la referencia la hace M2 en serio, con BLEU/ROUGE/BERTScore
# sobre este mismo split.


def entidades_inventadas(texto: str) -> list:
    return find_fabricated_entities(texto)


# Campos que debe traer cada registro guardado. Se usa para rechazar
# checkpoints de corridas con otro esquema (p. ej. los que traian 'similarity').
CAMPOS_ESPERADOS = {'generated', 'nombra_mecanismo', 'cita_no_verificable', 'entidades_inventadas'}


@torch.no_grad()
def generate_response(model, query: str) -> str:
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': query},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS_EVAL,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    generated = output[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


def evaluate(model, val_set, label: str, checkpoint: str):
    # Persiste CADA ejemplo a Drive en el momento de generarlo, y si el archivo
    # ya tiene resultados los reusa y sigue desde ahi. Antes la escritura
    # ocurria solo al terminar el lote: con MAX_NEW_TOKENS_EVAL en 900 la
    # evaluacion del baseline toma horas y una desconexion de Colab borraba
    # todo el trabajo de la corrida.
    import os
    import time

    ruta = f'{SALIDA_DIR}/{checkpoint}'
    results = []
    if os.path.exists(ruta):
        with open(ruta, encoding='utf-8') as f:
            results = [json.loads(linea) for linea in f if linea.strip()]
        # Guarda de identidad: en Drive quedan checkpoints de corridas
        # anteriores (otro adaptador, otro dataset, u otro esquema de campos).
        # Reusarlos mezclaria dos corridas en una sola cifra sin que nada falle.
        if len(results) > len(val_set):
            raise SystemExit(
                f'El checkpoint evaluacion/{checkpoint} tiene {len(results)} registros y '
                f'la validacion solo {len(val_set)}: es de otra corrida. Borralo de Drive.')
        if [r['id'] for r in results] != [item['id'] for item in val_set[:len(results)]]:
            raise SystemExit(
                f'El checkpoint evaluacion/{checkpoint} no corresponde a esta validacion '
                f'(ids distintos a los esperados). Es de una corrida anterior: borralo de Drive.')
        if results and not CAMPOS_ESPERADOS.issubset(results[0]):
            faltan = CAMPOS_ESPERADOS - set(results[0])
            raise SystemExit(
                f'El checkpoint evaluacion/{checkpoint} es de un esquema anterior (le faltan '
                f'{sorted(faltan)}). Borralo de Drive para medir todo con las metricas vigentes.')
        if results:
            print(f'[{label}] reusando {len(results)} resultados ya guardados '
                  f'en evaluacion/{checkpoint}', flush=True)
    ya_hechos = len(results)
    t0 = time.perf_counter()
    for i, item in enumerate(val_set, start=1):
        if i <= ya_hechos:
            continue
        query = item['messages'][1]['content']
        actual = generate_response(model, query)
        inventadas = entidades_inventadas(actual)
        fila = {
            **item,
            'generated': actual,
            'nombra_mecanismo': nombra_mecanismo(actual),
            'cita_no_verificable': cita_no_verificable(actual),
            'entidades_inventadas': inventadas,
            'palabras': len(actual.split()),
        }
        results.append(fila)
        with open(ruta, 'a', encoding='utf-8') as f:
            f.write(json.dumps(fila, ensure_ascii=False) + '\n')
        hechos = i - ya_hechos
        transcurrido = time.perf_counter() - t0
        eta = transcurrido / hechos * (len(val_set) - i) / 60
        print(f"[{label}] {i}/{len(val_set)}  "
              f"ruta={'si' if fila['nombra_mecanismo'] else 'no '}  "
              f"{transcurrido / hechos:.1f}s/ej  ETA ~{eta:.1f} min", flush=True)
    n = len(results)
    print(f"\n[{label}] ruta legal: {sum(r['nombra_mecanismo'] for r in results) / n:.1%}"
          f" | citas inventadas: {sum(r['cita_no_verificable'] for r in results) / n:.1%}"
          f" | entidades inventadas: {sum(1 for r in results if r['entidades_inventadas']) / n:.1%}"
          f" | {sum(r['palabras'] for r in results) / n:.0f} palabras")
    return results


def guardar_resultados(nombre, datos):
    ruta = f'{SALIDA_DIR}/{nombre}'
    with open(ruta, 'w', encoding='utf-8') as f:
        for r in datos:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')
    print(f'Guardado en Drive: evaluacion/{nombre} ({len(datos)} registros)')


## Paso 1 -- Baseline (modelo base, sin fine-tuning)

Esto puede tardar varios minutos segun el tamano de la validacion.

In [8]:
baseline_results = evaluate(model, val_records, label='baseline',
                            checkpoint='baseline_results.jsonl')
guardar_resultados('baseline_results.jsonl', baseline_results)

[baseline] 1/231  ruta=si  22.2s/ej  ETA ~85.1 min
[baseline] 2/231  ruta=si  18.9s/ej  ETA ~72.0 min
[baseline] 3/231  ruta=si  23.1s/ej  ETA ~87.8 min
[baseline] 4/231  ruta=si  22.2s/ej  ETA ~84.1 min
[baseline] 5/231  ruta=si  21.8s/ej  ETA ~82.2 min
[baseline] 6/231  ruta=si  21.7s/ej  ETA ~81.3 min
[baseline] 7/231  ruta=si  21.7s/ej  ETA ~80.9 min
[baseline] 8/231  ruta=si  21.5s/ej  ETA ~79.9 min
[baseline] 9/231  ruta=si  20.7s/ej  ETA ~76.7 min
[baseline] 10/231  ruta=si  20.6s/ej  ETA ~76.0 min
[baseline] 11/231  ruta=si  20.4s/ej  ETA ~74.8 min
[baseline] 12/231  ruta=si  20.9s/ej  ETA ~76.2 min
[baseline] 13/231  ruta=si  21.1s/ej  ETA ~76.6 min
[baseline] 14/231  ruta=si  20.1s/ej  ETA ~72.7 min
[baseline] 15/231  ruta=si  20.6s/ej  ETA ~74.3 min
[baseline] 16/231  ruta=si  20.5s/ej  ETA ~73.3 min
[baseline] 17/231  ruta=si  20.5s/ej  ETA ~73.2 min
[baseline] 18/231  ruta=si  20.4s/ej  ETA ~72.5 min
[baseline] 19/231  ruta=si  20.4s/ej  ETA ~72.0 min
[baseline] 20/231  ru

## Paso 2 -- Fine-tuning con LoRA

### Antes de entrenar: conecta Weights & Biases

M1 pide dejar registrada la curva de perdida del entrenamiento como evidencia visual de que el modelo esta aprendiendo. Crea una cuenta gratuita en wandb.ai si no tienes una, y ten a mano tu API key (la consigues en https://wandb.ai/authorize).

**Si corres este notebook desde VSCode** (extension de Colab) en vez del navegador: `wandb.login()` te pide la key en un recuadro de entrada; si se queda esperando sin mostrarlo, usa `wandb.login(key="tu_key")` en su lugar. Este notebook no depende de los Secrets de Colab (`userdata.get`), que es lo unico que no funciona fuera del navegador.

In [9]:
import wandb

wandb.login()  # pega tu API key cuando te la pida (solo la primera vez)

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: tomasposada67 (tomasposada67-universidad-eafit) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [10]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 10,092,544 || all params: 7,625,709,056 || trainable%: 0.1323


In [11]:
from datasets import Dataset

# Dev para elegir la epoca, recortado de TRAIN con el mismo metodo estratificado.
# NO se usa la validacion de 231 para esto: si el checkpoint se elige mirandola,
# deja de ser held-out y las cifras del reporte quedan contaminadas.
DEV_FRACTION = 0.08

random.seed(RANDOM_SEED)
_por_categoria = defaultdict(list)
for r in train_records:
    _por_categoria[r['category']].append(r)

fit_records, dev_records = [], []
for _categoria, _items in sorted(_por_categoria.items()):
    _items = _items[:]
    random.shuffle(_items)
    _n_dev = max(1, round(len(_items) * DEV_FRACTION))
    dev_records.extend(_items[:_n_dev])
    fit_records.extend(_items[_n_dev:])
random.shuffle(fit_records)
random.shuffle(dev_records)


# Formato prompt/completion: TRL enmascara el prompt y calcula la perdida SOLO
# sobre la respuesta del asistente.
#
# Antes se entrenaba con el texto ya renderizado (dataset_text_field='text'),
# asi que la perdida incluia el system prompt -- 87 palabras IDENTICAS en los
# 1536 ejemplos, el 60% del presupuesto -- mas la pregunta. Solo el 32% del
# gradiente caia sobre lo unico que el modelo debe aprender a generar, y lo que
# el modelo ajustaba con fuerza era el estilo.
#
# No se usa assistant_only_loss=True: ese camino necesita que el chat template
# traiga marcadores {% generation %} para construir la mascara del turno del
# asistente, y el de Qwen2.5-Instruct no los tiene. El formato prompt/completion
# no depende de la plantilla.
def a_prompt_completion(record):
    mensajes = record['messages']
    return {'prompt': mensajes[:2], 'completion': mensajes[2:]}


# El combinado trae ademas las columnas de v2 (modo, base_id, contexto,
# fuentes...), utiles para auditar pero no para entrenar. Se arma una copia
# delgada: Dataset.from_list tendria que inferir el esquema de 'contexto', que
# es una lista de diccionarios, y las columnas que sobran rompen el enmascarado
# de TRL (lo atrapa el assert de abajo, pero mejor no llegar ahi).
_PARA_SFT = ('id', 'category', 'messages')


def _delgado(registros):
    return [{k: r[k] for k in _PARA_SFT} for r in registros]


_COLUMNAS_ORIGINALES = list(_PARA_SFT)
train_dataset = Dataset.from_list(_delgado(fit_records)).map(
    a_prompt_completion, remove_columns=_COLUMNAS_ORIGINALES)
dev_dataset = Dataset.from_list(_delgado(dev_records)).map(
    a_prompt_completion, remove_columns=_COLUMNAS_ORIGINALES)

print(f'Entrenamiento: {len(train_dataset)} | Dev (eleccion de epoca): {len(dev_dataset)}')
print(f'Validacion intacta (solo para el reporte final): {len(val_records)}')
print('Columnas que ve TRL:', train_dataset.column_names)
assert train_dataset.column_names == ['prompt', 'completion'], (
    'TRL necesita exactamente las columnas prompt y completion para enmascarar '
    'el prompt. Con otras columnas vuelve a entrenar sobre el texto completo.')


Map:   0%|          | 0/1212 [00:00<?, ? examples/s]

Map:   0%|          | 0/93 [00:00<?, ? examples/s]

Entrenamiento: 1212 | Dev (eleccion de epoca): 93
Validacion intacta (solo para el reporte final): 231
Columnas que ve TRL: ['prompt', 'completion']


In [12]:
wandb.init(
    project='amparo-legal-finetune',
    name=f"lora-{MODEL_ID.split('/')[-1]}",
    config={
        'model_id': MODEL_ID,
        'lora_r': LORA_R,
        'lora_alpha': LORA_ALPHA,
        'epochs': NUM_EPOCHS,
        'learning_rate': LEARNING_RATE,
        'perdida': 'solo sobre la respuesta (prompt/completion)',
        'n_fit': len(train_dataset),
        'n_dev': len(dev_dataset),
    },
)

from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    logging_steps=10,
    # Evaluacion por epoca sobre el dev y se conserva el mejor checkpoint.
    # Antes no habia eval_dataset: solo se registraba la perdida de
    # entrenamiento, no habia forma de ver sobreajuste y las 3 epocas eran un
    # numero elegido, no medido -- se usaba el checkpoint de la ultima por
    # omision.
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    save_total_limit=2,
    bf16=True,
    max_length=MAX_SEQ_LENGTH,
    # Explicito: antes la reproducibilidad del entrenamiento dependia del
    # default de transformers, no de una decision escrita.
    seed=RANDOM_SEED,
    report_to='wandb',
    run_name=f"lora-{MODEL_ID.split('/')[-1]}",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    processing_class=tokenizer,
)
print('Trainer listo. Verifica el enmascarado en la celda siguiente ANTES de entrenar.')


Tokenizing train dataset:   0%|          | 0/1212 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1212 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1212 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1212 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/93 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/93 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/93 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/93 [00:00<?, ? examples/s]

Trainer listo. Verifica el enmascarado en la celda siguiente ANTES de entrenar.


### Verificacion: la perdida cae solo sobre la respuesta

In [13]:
# Verificacion del enmascarado, antes de gastar GPU.
#
# Esta celda existe porque el cambio a prompt/completion falla en silencio: si
# TRL no reconoce el formato, entrena sobre el texto completo igual que antes y
# el notebook corre sin un solo error. Lo unico que lo delata es la proporcion
# de tokens con etiqueta -100 (ignorados por la perdida).
import torch

_lote = next(iter(trainer.get_train_dataloader()))
_etiquetas = _lote['labels']
_ignorados = int((_etiquetas == -100).sum())
_aprendidos = int((_etiquetas != -100).sum())
_total = _ignorados + _aprendidos

print(f'Tokens del lote: {_total}')
print(f'  ignorados por la perdida (prompt): {_ignorados} ({100 * _ignorados / _total:.0f}%)')
print(f'  con perdida (respuesta):           {_aprendidos} ({100 * _aprendidos / _total:.0f}%)')

# Si el prompt NO se esta enmascarando, casi nada queda en -100 (solo el
# padding), y eso es exactamente el bug que esta celda busca.
assert _ignorados > _aprendidos, (
    f'El prompt no se esta enmascarando: solo {100 * _ignorados / _total:.0f}% de los '
    'tokens quedan fuera de la perdida. Con el system prompt de 87 palabras se '
    'espera que la mayoria del lote este ignorada. Revisa que train_dataset '
    'tenga solo las columnas prompt y completion.')
print('\nOK: la perdida se calcula solo sobre la respuesta del asistente.')


Tokens del lote: 534
  ignorados por la perdida (prompt): 380 (71%)
  con perdida (respuesta):           154 (29%)

OK: la perdida se calcula solo sobre la respuesta del asistente.


### Entrenar

In [14]:
trainer.train()

print('Dashboard de W&B (guarda este link para tu informe de M1):', wandb.run.url)
wandb.finish()


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,1.666431,1.676553,1.669447,316381.000000,0.602089
2,1.483398,1.590809,1.501894,632762.000000,0.615108
3,1.338528,1.579355,1.445845,949143.000000,0.619051


Dashboard de W&B (guarda este link para tu informe de M1): https://wandb.ai/tomasposada67-universidad-eafit/amparo-legal-finetune/runs/z2tc1rda


eval/entropy,█▃▁
eval/loss,█▂▁
eval/mean_token_accuracy,▁▆█
eval/num_tokens,▁▅█
eval/runtime,█▆▁
eval/samples_per_second,▁▃█
eval/steps_per_second,▁██
train/entropy,▄█▅▄▄▄▄▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁
train/epoch,▁▁▂▂▂▃▃▃▃▄▄▄▅▅▅▆▆▆▆▆▇▇▇███
train/global_step,▁▁▂▂▂▃▃▃▃▄▄▄▅▅▅▅▆▆▆▆▇▇▇███
+5,...


## Guardar el adaptador LoRA en Drive

Se guarda directo en tu Drive (no solo son unos MB, son los pesos LoRA) para que no se pierda si el entorno de Colab se desconecta a mitad del entrenamiento o despues.

In [15]:
import shutil

model.save_pretrained(f'{OUTPUT_DIR}/adapter')
tokenizer.save_pretrained(f'{OUTPUT_DIR}/adapter')

# ADAPTER_DIR viene de la celda de configuracion de Drive: una sola ruta,
# definida en un solo lugar, para que no se desincronice con el resto.
shutil.copytree(f'{OUTPUT_DIR}/adapter', ADAPTER_DIR, dirs_exist_ok=True)
print(f'Adaptador guardado en {ADAPTER_DIR}')

# Copia fechada: la carpeta sin version se sobrescribe en cada corrida, y por eso
# el adaptador del 27 de septiembre ya no existe. Con esta queda el respaldo.
shutil.copytree(f'{OUTPUT_DIR}/adapter', ADAPTER_DIR_VERSIONADO, dirs_exist_ok=True)
print(f'Respaldo fechado en {ADAPTER_DIR_VERSIONADO}')

Adaptador guardado en /content/drive/MyDrive/Colab Notebooks/Amparo/amparo-lora-adapter


In [16]:
# prepare_model_for_kbit_training() dejo gradient checkpointing activo, lo que fuerza use_cache=False
# durante model.generate() (recalcula toda la atencion en cada token, sin KV-cache). Sin esto, el
# Paso 3 tarda varias veces mas de lo necesario y puede agotar la cuota de GPU de Colab a mitad de la evaluacion.
model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(152064, 3584)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3584, out_features=3584, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3584, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3584, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

## Paso 3 -- Evaluar el modelo ya afinado (misma validacion)

In [17]:
finetuned_results = evaluate(model, val_records, label='fine-tuned',
                             checkpoint='finetuned_results.jsonl')
guardar_resultados('finetuned_results.jsonl', finetuned_results)

[fine-tuned] 1/231  ruta=si  6.0s/ej  ETA ~23.1 min
[fine-tuned] 2/231  ruta=si  6.6s/ej  ETA ~25.1 min
[fine-tuned] 3/231  ruta=si  6.6s/ej  ETA ~25.3 min
[fine-tuned] 4/231  ruta=si  6.6s/ej  ETA ~25.1 min
[fine-tuned] 5/231  ruta=si  6.7s/ej  ETA ~25.4 min
[fine-tuned] 6/231  ruta=si  6.5s/ej  ETA ~24.4 min
[fine-tuned] 7/231  ruta=si  6.4s/ej  ETA ~24.0 min
[fine-tuned] 8/231  ruta=si  6.5s/ej  ETA ~24.2 min
[fine-tuned] 9/231  ruta=si  6.5s/ej  ETA ~24.2 min
[fine-tuned] 10/231  ruta=si  6.5s/ej  ETA ~24.1 min
[fine-tuned] 11/231  ruta=si  6.6s/ej  ETA ~24.1 min
[fine-tuned] 12/231  ruta=si  6.7s/ej  ETA ~24.4 min
[fine-tuned] 13/231  ruta=si  6.7s/ej  ETA ~24.2 min
[fine-tuned] 14/231  ruta=si  6.6s/ej  ETA ~24.0 min
[fine-tuned] 15/231  ruta=si  6.7s/ej  ETA ~24.0 min
[fine-tuned] 16/231  ruta=si  6.7s/ej  ETA ~24.1 min
[fine-tuned] 17/231  ruta=si  6.8s/ej  ETA ~24.1 min
[fine-tuned] 18/231  ruta=si  6.7s/ej  ETA ~23.9 min
[fine-tuned] 19/231  ruta=si  6.7s/ej  ETA ~23.8 min
[f

## Paso 4 -- Comparacion baseline vs. afinado

**Como leerla.** Cada proporcion va con su IC95 de Wilson y la diferencia lleva
`*` cuando la prueba pareada no incluye el 0. Sin eso, cinco ejemplos de 231
parecen un hallazgo.

Dos trampas concretas de esta tabla:

- **"Nombra ruta legal" esta saturada y confundida con la longitud.** El modelo
  base ya venia en 96%, y como la metrica es coincidencia de palabras clave, el
  baseline (que escribe ~5x mas texto) tiene ~5x mas superficie para acertar por
  azar. No sirve para demostrar mejora en ninguna direccion.
- **El desglose por categoria imprime el `n`.** Nueve de las 33 categorias
  tienen `val<=4`, y ahi un "+33%" es UN ejemplo.

Lo que si soporta una conclusion son los dos modos de fabricacion: citas
inventadas y entidades inventadas.


In [18]:
import pandas as pd

from tools.evaluation.estadistica import diferencia_pareada, proporcion

# Las banderas se recalculan aqui desde el texto generado, en vez de leer las
# que quedaron guardadas en cada registro: si una metrica se corrige despues de
# una corrida, esta tabla refleja la definicion vigente y no una medicion vieja.
filas = []
for etiqueta, datos in [('baseline', baseline_results), ('fine_tuned', finetuned_results)]:
    for r in datos:
        filas.append({'id': r['id'], 'category': r['category'], 'modelo': etiqueta,
                      'nombra_mecanismo': nombra_mecanismo(r['generated']),
                      'cita_no_verificable': cita_no_verificable(r['generated']),
                      'entidad_inventada': bool(entidades_inventadas(r['generated'])),
                      'palabras': len(r['generated'].split())})
df = pd.DataFrame(filas)

N = len(baseline_results)
pares = {}
for col in ('nombra_mecanismo', 'cita_no_verificable', 'entidad_inventada'):
    a = df[df.modelo == 'baseline'].sort_values('id')[col].astype(float).tolist()
    b = df[df.modelo == 'fine_tuned'].sort_values('id')[col].astype(float).tolist()
    pares[col] = diferencia_pareada(a, b)

print('=' * 78)
print(f'RESULTADO PRINCIPAL -- metricas de dominio (N = {N} de validacion)')
print('=' * 78)
print(f"{'metrica':<30}{'baseline':>18}{'afinado':>18}{'diferencia':>12}")
print('-' * 78)
for col, nombre in (('nombra_mecanismo', 'Nombra ruta legal'),
                    ('cita_no_verificable', 'Cita normas inventadas'),
                    ('entidad_inventada', 'Inventa entidades')):
    k_b = int(df[(df.modelo == 'baseline')][col].sum())
    k_f = int(df[(df.modelo == 'fine_tuned')][col].sum())
    iv_b, iv_f, dif = proporcion(k_b, N), proporcion(k_f, N), pares[col]
    marca = '*' if dif.significativo else ' '
    print(f'{nombre:<30}'
          f'{f"{100*k_b/N:.1f}% [{100*iv_b.bajo:.0f}-{100*iv_b.alto:.0f}]":>18}'
          f'{f"{100*k_f/N:.1f}% [{100*iv_f.bajo:.0f}-{100*iv_f.alto:.0f}]":>18}'
          f'{f"{100*dif.valor:+.1f}{marca}":>12}')
print(f"{'Longitud media (palabras)':<30}"
      f"{df[df.modelo=='baseline'].palabras.mean():>18.0f}"
      f"{df[df.modelo=='fine_tuned'].palabras.mean():>18.0f}")
print('-' * 78)
print('[x-y] = IC95 de Wilson.  (*) = la diferencia pareada no incluye el 0.')
print('La similitud contra la referencia se mide en M2 (BLEU/ROUGE/BERTScore),')
print('sobre este mismo split. Aqui no se duplica.')
print('=' * 78)

# --- Desglose por categoria -------------------------------------------------
# Se imprime el n de cada categoria a proposito: 9 de las 33 tienen val<=4, y
# ahi un "+33%" es UN ejemplo. Sin el denominador a la vista, esa columna
# invita a leer ruido como efecto.
por_cat = (df.pivot_table(index='category', columns='modelo', values='nombra_mecanismo', aggfunc='mean')
             .reindex(columns=['baseline', 'fine_tuned']) * 100).round(0)
por_cat['n_val'] = df[df.modelo == 'baseline'].groupby('category').size()
por_cat['interpretable'] = por_cat['n_val'].map(lambda n: 'si' if n >= 8 else 'n muy chico')
print('\nNombra una ruta legal, por categoria:')
print(por_cat.sort_values('n_val').to_string())

df.to_csv(f'{SALIDA_DIR}/metricas_por_registro.csv', index=False)
por_cat.to_csv(f'{SALIDA_DIR}/metricas_por_categoria.csv')
print(f'\nCSV guardados en Drive: evaluacion/')


RESULTADO PRINCIPAL -- metricas de dominio (N = 231 de validacion)
metrica                                 baseline           afinado  diferencia
------------------------------------------------------------------------------
Nombra ruta legal                  96.1% [93-98]     93.5% [90-96]       -2.6 
Cita normas inventadas             14.7% [11-20]        0.0% [0-2]      -14.7*
Inventa entidades                   11.7% [8-16]        3.0% [1-6]       -8.7*
Longitud media (palabras)                    229                45
------------------------------------------------------------------------------
[x-y] = IC95 de Wilson.  (*) = la diferencia pareada no incluye el 0.
La similitud contra la referencia se mide en M2 (BLEU/ROUGE/BERTScore),
sobre este mismo split. Aqui no se duplica.

Nombra una ruta legal, por categoria:
modelo                                           baseline  fine_tuned  n_val interpretable
category                                                                    

## Paso 4b -- Ejemplos reales de respuestas

Las cifras de arriba no se pueden juzgar sin ver que generó el modelo. Esta celda
imprime casos concretos (pregunta, respuesta de referencia, baseline y afinado) y
guarda una tabla en Drive para citarla en la documentación. Se muestran tanto
casos donde el fine-tuning mejoró como casos donde no, para no documentar solo lo
que conviene.

In [19]:
import pandas as pd

pares = list(zip(baseline_results, finetuned_results))
mejoro = [(b, f) for b, f in pares if f['nombra_mecanismo'] and not b['nombra_mecanismo']]
sin_ruta = [(b, f) for b, f in pares if not f['nombra_mecanismo']]
# Los dos modos de fabricacion, que son el resultado real de M1. Leer estos
# casos es la unica forma de ver QUE se inventa: la cifra sola no lo dice.
cito_normas = [(b, f) for b, f in pares if cita_no_verificable(f['generated'])]
invento_entidad = [(b, f) for b, f in pares if entidades_inventadas(f['generated'])]


def mostrar(titulo, casos, n=3, con_entidades=False):
    print('=' * 78)
    print(titulo)
    print('=' * 78)
    for b, f in casos[:n]:
        print(f"\nCATEGORIA: {b['category']}  (id {b['id']})")
        print(f"PREGUNTA:   {b['messages'][1]['content']}")
        print(f"REFERENCIA: {b['messages'][2]['content']}")
        print(f"BASELINE:   {b['generated']}")
        print(f"AFINADO:    {f['generated']}")
        if con_entidades:
            print(f"  -> marcadas en el afinado: {entidades_inventadas(f['generated'])}")
            print(f"  -> marcadas en el baseline: {entidades_inventadas(b['generated'])}")


mostrar(f'CASOS DONDE EL FINE-TUNING APORTO ({len(mejoro)} en total)', mejoro)
mostrar(f'CASOS DONDE EL AFINADO SIGUE SIN NOMBRAR UNA RUTA ({len(sin_ruta)} en total)', sin_ruta)
mostrar(f'EL AFINADO SE INVENTO UNA ENTIDAD ({len(invento_entidad)} en total)',
        invento_entidad, n=5, con_entidades=True)
mostrar(f'EL AFINADO CITO UNA NORMA POR NUMERO ({len(cito_normas)} en total)', cito_normas, n=5)

tabla = pd.DataFrame([{
    'id': b['id'], 'categoria': b['category'],
    'pregunta': b['messages'][1]['content'],
    'referencia': b['messages'][2]['content'],
    'baseline': b['generated'], 'afinado': f['generated'],
    'baseline_ruta': b['nombra_mecanismo'], 'afinado_ruta': f['nombra_mecanismo'],
    'baseline_entidades': entidades_inventadas(b['generated']),
    'afinado_entidades': entidades_inventadas(f['generated']),
    'baseline_cito': cita_no_verificable(b['generated']),
    'afinado_cito': cita_no_verificable(f['generated']),
} for b, f in pares])
tabla.to_csv(f'{SALIDA_DIR}/ejemplos_baseline_vs_afinado.csv', index=False)
print(f'\n\nTabla completa guardada en Drive: evaluacion/ejemplos_baseline_vs_afinado.csv')


CASOS DONDE EL FINE-TUNING APORTO (7 en total)

CATEGORIA: Peticiones de conducta ilegitima  (id 1494)
PREGUNTA:   Puedo vender el carro que esta embargado sin decirle al comprador?
REFERENCIA: No: vender un bien con medidas vigentes y sin informarlo puede ser estafa, y el comprador podría demandarte. Pide primero en el juzgado el levantamiento del embargo, o informa la situación y acuerden el precio con eso a la vista. Pide el certificado de tradición del vehículo en el RUNT.
BASELINE:   No puedes vender un vehículo que está embargado sin informarle al comprador. El embargo judicial significa que el vehículo está bajo control del juez debido a una orden judicial. Venderlo sin el conocimiento del comprador y sin el consentimiento del juez sería fraudulento y puede ser considerado como un delito.

Para resolver esta situación, debes:

1. Informar al comprador sobre el embargo.
2. Buscar la autorización del juez para la venta.
3. Realizar la venta solo con la autorización judicial.

Si e

## Paso 4c -- Publicar el adaptador en Hugging Face Hub

El adaptador no puede vivir solo en un Drive privado: sin acceso a ese Drive nadie
puede verificar los resultados ni reproducirlos. Subirlo al Hub lo vuelve
auditable y permite cargarlo desde cualquier parte con su identificador.

Necesitas un token de escritura de https://huggingface.co/settings/tokens.

In [21]:
from huggingface_hub import login, create_repo, upload_folder

HF_USER = 'Tomas0626'          # ajusta si tu usuario es otro
REPO_ID = f'{HF_USER}/amparo-lora-qwen2.5-7b'

login()   # pega tu token de escritura cuando lo pida

create_repo(REPO_ID, repo_type='model', exist_ok=True)
upload_folder(
    repo_id=REPO_ID,
    folder_path=ADAPTER_DIR,
    commit_message=f'Adaptador LoRA de M1 ({VERSION_ADAPTADOR}, dataset v1+v2: 2291 ejemplos, 33 categorias)',
)
print(f'Adaptador publicado: https://huggingface.co/{REPO_ID}')

No files have been modified since last commit. Skipping to prevent empty commit.


Adaptador publicado: https://huggingface.co/Tomas0626/amparo-lora-qwen2.5-7b
